In [50]:
import amulet
import os
import sys
import json
import numpy as np
import pandas as pd
from amulet import load_level
from amulet_nbt import load
from amulet.api.selection import SelectionBox
from collections import namedtuple
import re
import mcschematic
from gdpc import Block
from gdpc.block import transformedBlockOrPalette
import itertools


In [2]:
data_path = '../../data/'
palette = 'java_palette_copy_2.json'
palette_path = os.path.join(data_path, palette)

In [74]:
class Palette:
    
    def __init__(self, src: str | dict | list):
        # Get the block to token mapping
        if isinstance(src, str):
            with open(src, 'r') as file:
                self.block2token = json.load(file)
        elif isinstance(src, dict):
            self.block2token = src
        elif isinstance(src, list):
            self.block2token = {k : i for i, k in enumerate(src)}
        else:
            raise ValueError("src argument should be a file path (str), block2token mapping (dict), or list of blocks (list)")

        # Validate that all block strings are valid
        self._blockstr_pattern = re.compile(r"^minecraft:[a-z0-9_]+(?:\[[a-z0-9_]+=[a-z0-9_]+(?:,[a-z0-9_]+=[a-z0-9_]+)*\])?$")
        self._blockstr_separation_pattern = pattern = re.compile(r"(minecraft:[^\[]+)(?:\[(.*)\])?")
        
        invalid_blocks = []
        for blockstr in self.block2token.keys():
            if not self._valid_blockstr(blockstr):
                invalid_blocks.append(blockstr)
        
        if invalid_blocks:
            raise ValueError(f"The following block strings are invalid: {invalid_blocks}")
            
        # Get other mapping lists
        self.block_strings = [blockstr for blockstr in self.block2token.keys()]
        self.token2block = {v:k for k,v in self.block2token.items()}
        self.gdpc_blocks = [self._blockstr_to_gdpc_block(blockstr) for blockstr in self.block_strings]
        self.length = len(self.block_strings)
        
    def reduce_blockstates(self, keep_blockstates: list) -> tuple["Palette", np.ndarray, np.ndarray]:
        reduced_block2tok = {}
        reduced_gdpc_blocks = []
        src2tgt_lookup = np.zeros(self.length, dtype=np.int16)
        
        for blockstr, token in self.block2token.items():
            block_id, states = self._blockstr_to_id_states(blockstr)
            
            # remove any blockstate info were not keeping
            if states: states = {k:v for k,v in states.items() if k in keep_blockstates}
            
            # Using GDPC block objects since they can tell if two blocks with different state orders are equal
            reduced_block = Block(block_id, states)
            
            # add to the new palette if its not already in
            if reduced_block not in reduced_gdpc_blocks:
                reduced_gdpc_blocks.append(reduced_block)
                reduced_block2tok[str(reduced_block)] = len(reduced_block2tok)
            
            # Now, map this block to the reduced palette
            reduced_token = reduced_block2tok[str(reduced_block)]
            src2tgt_lookup[token] = reduced_token
        
        # Now, we reverse the lookup table so we can convert back to the original palette. The tgt tokens will map to the first instance in the src that maps to it
        tgt2src_lookup = np.full(len(reduced_block2tok), -1, dtype=np.int16)
        
        for i, token in enumerate(src2tgt_lookup):
            if tgt2src_lookup[token] == -1:
                tgt2src_lookup[token] = i
        
        # Now, construct the new Palette
        reduced_palette = Palette(reduced_block2tok)
        
        return reduced_palette, src2tgt_lookup, tgt2src_lookup
    
    def _generate_transformation_lookups(self):
        
        pass
    
    def _add_missing_transformations(self):
        count_added = 0
        rotations = [0,1,2,3]
        flips = [0,1]
        combos = list(itertools.product(rotations, flips))
        print(combos)
        for rotation, flip in combos:
            all_possible_blocks = transformedBlockOrPalette(block=self.gdpc_blocks, rotation=rotation, flip=(flip,0,0))
            for block in all_possible_blocks:
                if block not in self.gdpc_blocks:
                    blockstr = str(block)
                    self.gdpc_blocks.append(block)
                    self.block_strings.append(blockstr)
                    self.block2token[blockstr] = len(self.block2token)
                    self.token2block[self.block2token[blockstr]] = blockstr
                    count_added += 1
        self.length = len(self.block_strings)
        print(f'Added {count_added} new blocks')
                
    
    def _valid_blockstr(self, blockstr: str) -> bool:
        return bool(self._blockstr_pattern.fullmatch(blockstr))
    
    def _blockstr_to_id_states(self, blockstr: str):
        block_id, states_str = self._blockstr_separation_pattern.fullmatch(blockstr).groups()
        states = dict(p.split("=") for p in states_str.split(",")) if states_str else None
        return block_id, states
    
    def _blockstr_to_gdpc_block(self, blockstr):
        block_id, states = self._blockstr_to_id_states(blockstr)
        return Block(block_id, states)
    
    def __str__(self):
        lines = [f"{token}: {block}" for block, token in self.block2token.items()]
        return '\n'.join(lines)
    

In [75]:
java_palette = Palette(palette_path)
java_palette._add_missing_transformations()

keep_blockstates = ["axis", "facing", "shape", "east", "west", "north", "south", "face", "half"]
keep_blockstates = ["axis", "facing", "shape"]

[(0, 0), (0, 1), (1, 0), (1, 1), (2, 0), (2, 1), (3, 0), (3, 1)]
Added 738 new blocks


In [70]:
print(java_palette)

0: minecraft:air
1: minecraft:dirt
2: minecraft:grass_block[snowy=false]
3: minecraft:coarse_dirt
4: minecraft:green_concrete
5: minecraft:green_wool
6: minecraft:cyan_terracotta
7: minecraft:oak_leaves[distance=7,persistent=true]
8: minecraft:dark_oak_leaves[distance=7,persistent=false]
9: minecraft:green_glazed_terracotta[facing=south]
10: minecraft:grass
11: minecraft:spruce_planks
12: minecraft:oak_log[axis=x]
13: minecraft:brick_stairs[facing=north,half=bottom,shape=inner_left]
14: minecraft:brick_stairs[facing=west,half=bottom,shape=straight]
15: minecraft:brick_stairs[facing=west,half=bottom,shape=inner_left]
16: minecraft:brick_stairs[facing=north,half=bottom,shape=straight]
17: minecraft:brick_stairs[facing=south,half=bottom,shape=straight]
18: minecraft:brick_stairs[facing=east,half=bottom,shape=inner_left]
19: minecraft:brick_stairs[facing=east,half=bottom,shape=straight]
20: minecraft:brick_stairs[facing=east,half=bottom,shape=inner_right]
21: minecraft:oak_slab[type=bottom

In [76]:
reduced_palette, src2tgt, tgt2src = java_palette.reduce_blockstates(keep_blockstates)

In [77]:
print(len(src2tgt))
print(len(tgt2src))


7600
2615


In [ ]:
schem_name = 'build_batch_65_1671_1.schem'


schem_path = os.path.join(data_path, 'processed_builds/', schem_name)

AttributeError: 'str' object has no attribute 'transformed'

In [65]:
block_str = "minecraft:grass_block[snowy=false]"
block_str = "minecraft:birch_button[face=wall,facing=south,powered=false]"
block_str = "minecraft:oxidized_cut_copper_stairs[facing=south,half=bottom,shape=inner_left]"
block_str2 = "minecraft:oxidized_cut_copper_stairs[half=bottom,facing=south,shape=inner_left]"

pattern = re.compile(r"(minecraft:[^\[]+)(?:\[(.*)\])?")
block_id, states_str = pattern.fullmatch(block_str).groups()
states = dict(p.split("=") for p in states_str.split(",")) if states_str else None

keep_blockstates = ['axis', 'facing', 'shape']

block_obj = Block(block_id, states, None)

block_id2, states_str2 = pattern.fullmatch(block_str2).groups()
states2 = dict(p.split("=") for p in states_str2.split(",")) if states_str2 else None

block_obj2 = Block(block_id2, states2, None)


In [68]:
block_obj == block_obj2

True

In [66]:
print(block_obj.id)
print(block_obj.states)
print(block_obj.data)

minecraft:oxidized_cut_copper_stairs
{'facing': 'south', 'half': 'bottom', 'shape': 'inner_left'}
None


In [ ]:
print(block_obj2.id)
print(block_obj2.states)
print(block_obj2.data)

minecraft:oxidized_cut_copper_stairs
{'half': 'bottom', 'facing': 'south', 'shape': 'inner_left'}
None


In [59]:
block_transformed = block_obj.transformed(0,(0,0,1))

In [75]:
print(block_transformed.id)
print(block_transformed.states)
print(block_transformed.data)
print(block_obj)

minecraft:oxidized_cut_copper_stairs
{'facing': 'north', 'half': 'bottom', 'shape': 'inner_left'}
None
minecraft:oxidized_cut_copper_stairs[facing=south,half=bottom,shape=inner_left]
